# 🛡️ File 1: Data Preparation Pipeline (Ekstraksi, Pembersihan, & Domain-Level Splitting)
**Riset Skripsi**: Perbandingan Kinerja RoBERTa-base dan DistilBERT-base untuk Deteksi Phishing URL  
- **Target Runtime**: Google Colab (GPU NVIDIA Tesla T4 16GB / CUDA 12.x)  
- **Storage Backend**: Google Drive (`/content/drive/MyDrive/Skripsi/`)  
- **Tujuan Utama**: Menghasilkan dataset bersih bebas dari duplikat, label konflik, dan kebocoran domain (*zero data leakage*) dengan skema `GroupShuffleSplit`.

---
### 📋 Ringkasan Alur Kerja Notebook:
1. **Setup Lingkungan & Mount Google Drive**: Menghubungkan Google Drive untuk persistensi data dan mengimpor pustaka.
2. **Pemuatan Dataset Mentah**: Membaca file serialisasi `.dill` atau `vonDataset_clean.csv`.
3. **Normalisasi & Pembersihan Duplikasi**: Lowercasing, pemotongan skema dan prefix www, eliminasi konflik label, serta drop duplikat laten.
4. **Ekstraksi Domain & Penanganan Shared Hosting**: Mengekstrak registered domain dengan `tldextract` dan menangani shared hosting / shortener via path tingkat pertama.
5. **Domain-Level Splitting (70:15:15)**: Pemisahan data latih, validasi, dan uji berbasis grup domain (`GroupShuffleSplit`).
6. **Zero-Leak Verification Test**: Serangkaian pengujian asersi ketat untuk memastikan tidak ada domain atau URL yang bocor antar partisi.
7. **Penyimpanan Permanen**: Menyimpan `train.csv`, `val.csv`, dan `test.csv` ke Google Drive.


## 1. Setup Lingkungan & Instalasi Dependensi
Menginstalasi modul `dill`, `tldextract`, `pandas`, `scikit-learn`, `tqdm` untuk pemrosesan teks URL dan splitting data.


In [3]:
# Instalasi library yang dibutuhkan di Google Colab
!pip install dill tldextract pandas numpy tqdm scikit-learn -q

import os
import sys
import gc
import time
import re
import dill
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import tldextract
from sklearn.model_selection import GroupShuffleSplit
from google.colab import drive

print("[OK] Seluruh pustaka pendukung berhasil diimpor.")


[OK] Seluruh pustaka pendukung berhasil diimpor.


## 2. Mount Google Drive & Konfigurasi Path Penyimpanan
Menghubungkan direktori Google Drive (`/content/drive/MyDrive/Skripsi`) dan menyiapkan jalur input serta output.


In [4]:
# 1. Mount Google Drive
print("[INFO] Melakukan mounting Google Drive...")
drive.mount('/content/drive')

# 2. Konfigurasi path direktori kerja
BASE_DIR = "/content/drive/MyDrive/Skripsi"
DATA_DIR = os.path.join(BASE_DIR, "new data")
os.makedirs(DATA_DIR, exist_ok=True)

# 3. Path dataset input (prioritas CSV bersih, fallback file .dill mentah)
CLEAN_CSV_PATH = os.path.join(DATA_DIR, "vonDataset_clean.csv")
CLEAN_CSV_ALT = os.path.join(BASE_DIR, "vonDataset_clean.csv")
DILL_PATH_PRIMARY = os.path.join(BASE_DIR, "vonDataset20180426.dill")
DILL_PATH_ALT = os.path.join(DATA_DIR, "vonDataset20180426.dill")

# 4. Path dataset target output
TRAIN_CSV_PATH = os.path.join(DATA_DIR, "train.csv")
VAL_CSV_PATH = os.path.join(DATA_DIR, "val.csv")
TEST_CSV_PATH = os.path.join(DATA_DIR, "test.csv")

print(f"[OK] Direktori Utama : {BASE_DIR}")
print(f"[OK] Direktori Data  : {DATA_DIR}")


Mounted at /content/drive
[OK] Direktori Utama : /content/drive/MyDrive/Skripsi
[OK] Direktori Data  : /content/drive/MyDrive/Skripsi/new data


## 3. Ekstraksi Dataset Mentah (.dill / CSV)
Memuat data dari Google Drive. Jika `vonDataset_clean.csv` telah tersedia, data dimuat secara langsung. Jika file `.dill` yang tersedia, integer sequence didekode menjadi URL teks asli.


In [6]:
def load_raw_dataset():
    '''
    Memuat dataset dari CSV bersih jika sudah ada, atau mendekode dari file .dill mentah.
    Mengembalikan pandas DataFrame dengan kolom ['url', 'label'].
    '''
    # 1. Cek keberadaan file CSV bersih
    for csv_path in [CLEAN_CSV_PATH, CLEAN_CSV_ALT]:
        if os.path.exists(csv_path):
            print(f"[INFO] Ditemukan file CSV bersih: {csv_path}")
            print(f"[INFO] Membaca data...")
            df = pd.read_csv(csv_path, usecols=['url', 'label'])
            print(f"[OK] Berhasil memuat {len(df):,} baris dari CSV.")
            return df

    # 2. Jika CSV belum ada, muat dari file .dill
    dill_target = None
    for d_path in [DILL_PATH_PRIMARY, DILL_PATH_ALT]:
        if os.path.exists(d_path):
            dill_target = d_path
            break

    if dill_target is not None:
        print(f"[INFO] Membaca file serialisasi .dill dari: {dill_target} (~20-45 dtk)...")
        t0 = time.perf_counter()
        with open(dill_target, "rb") as f:
            raw_data = dill.load(f)
        print(f"[OK] File .dill dimuat dalam {time.perf_counter() - t0:.2f} detik.")

        # Bangun pemetaan int -> char
        char_to_int = raw_data.get("char_to_int", {})
        int_to_char = {idx: ch for ch, idx in char_to_int.items() if ch not in ('<PAD>', '<pad>', '<nul>', '', '\x00', None)}

        max_idx = max(int_to_char.keys()) if int_to_char else 256
        for split_k in ["train_x", "val_x", "test_x"]:
            if split_k in raw_data:
                max_idx = max(max_idx, int(np.max(raw_data[split_k][:500])))

        lookup_size = max_idx + 20
        lookup_table = [""] * lookup_size
        for idx, ch in int_to_char.items():
            if idx < lookup_size:
                lookup_table[idx] = ch

        def decode_seq(seq):
            return "".join(lookup_table[c] for c in seq if c < lookup_size and lookup_table[c]).replace("补", "").strip()

        urls_list = []
        labels_list = []

        for x_key, y_key in [("train_x", "train_y"), ("val_x", "val_y"), ("test_x", "test_y")]:
            if x_key in raw_data and y_key in raw_data:
                print(f"[INFO] Mendekode partisi {x_key} ({len(raw_data[y_key]):,} sampel)...")
                X = raw_data[x_key]
                y = raw_data[y_key]
                for i in tqdm(range(len(y)), desc=f"Decode {x_key}"):
                    decoded_url = decode_seq(X[i])
                    urls_list.append(decoded_url)
                    labels_list.append(int(y[i]))

        df = pd.DataFrame({"url": urls_list, "label": labels_list})
        del raw_data, urls_list, labels_list
        gc.collect()
        return df

    raise FileNotFoundError(
        f"❌ File dataset tidak ditemukan di Drive!\n"
        f"Harap pastikan salah satu file berikut berada di Google Drive:\n"
        f"- {CLEAN_CSV_PATH}\n"
        f"- {DILL_PATH_PRIMARY}"
    )

df_raw = load_raw_dataset()

# Filter awal nilai NaN atau URL kosong
df_raw = df_raw.dropna(subset=['url', 'label']).copy()
df_raw['url'] = df_raw['url'].astype(str).str.strip()
df_raw = df_raw[df_raw['url'] != ''].copy()
df_raw['label'] = pd.to_numeric(df_raw['label'], errors='coerce')
df_raw = df_raw[df_raw['label'].isin([0, 1])].copy()
df_raw['label'] = df_raw['label'].astype(int)

print(f"\n[INFO] Dataset mentah tervalidasi: {len(df_raw):,} baris.")
print("Distribusi Label Awal:")
print(df_raw['label'].value_counts())
print(df_raw['label'].value_counts(normalize=True) * 100)


[INFO] Membaca file serialisasi .dill dari: /content/drive/MyDrive/Skripsi/vonDataset20180426.dill (~20-45 dtk)...
[OK] File .dill dimuat dalam 14.02 detik.
[INFO] Mendekode partisi train_x (1,247,489 sampel)...


Decode train_x:   0%|          | 0/1247489 [00:00<?, ?it/s]

[INFO] Mendekode partisi val_x (155,936 sampel)...


Decode val_x:   0%|          | 0/155936 [00:00<?, ?it/s]

[INFO] Mendekode partisi test_x (155,937 sampel)...


Decode test_x:   0%|          | 0/155937 [00:00<?, ?it/s]


[INFO] Dataset mentah tervalidasi: 1,559,362 baris.
Distribusi Label Awal:
label
0    800000
1    759362
Name: count, dtype: int64
label
0    51.303033
1    48.696967
Name: proportion, dtype: float64


## 4. Normalisasi URL & Pembersihan Duplikasi / Konflik Label
1. **Normalisasi**:
   - Huruf kecil (*lowercase*).
   - Pemotongan skema protokol (`http://`, `https://`, dll.).
   - Pemotongan prefix `www.` atau variasinya (`www1.`, `www2.`).
   - Pemotongan *trailing slash* (`/`).
2. **Eliminasi Konflik Label**: Menghapus seluruh URL identik yang terlabel ganda (0 dan 1 sekaligus).
3. **Deduplikasi Laten**: Menghapus URL duplikat berdasarkan `url_norm`.


In [7]:
def normalize_url(url: str) -> str:
    '''
    Normalisasi URL standar:
    - Lowercase
    - Hapus skema (http://, https://)
    - Hapus www.
    - Hapus trailing slashes
    '''
    if not isinstance(url, str):
        return ""
    u = url.strip().lower()
    # Hapus skema protokol
    u = re.sub(r'^[a-z]+://', '', u)
    # Hapus prefix www
    u = re.sub(r"^www\d*\.", "", u)
    # Hapus trailing slashes berulang di akhir
    u = re.sub(r'/+$', '', u)
    return u.strip()

print("[INFO] Menjalankan fungsi normalisasi URL...")
df_raw['url_norm'] = df_raw['url'].apply(normalize_url)

# Pastikan URL normal tidak kosong
df_raw = df_raw[df_raw['url_norm'] != ''].reset_index(drop=True)
print(f"[OK] Baris setelah normalisasi: {len(df_raw):,}")

# Deteksi dan hapus seluruh URL yang mengalami konflik label (label 0 dan 1 sekaligus)
print("\n[INFO] Mendeteksi URL dengan konflik label...")
url_label_nunique = df_raw.groupby('url_norm')['label'].transform('nunique')
conflict_mask = url_label_nunique > 1
n_conflicts = conflict_mask.sum()

if n_conflicts > 0:
    print(f"[PERINGATAN] Ditemukan {n_conflicts:,} baris URL dengan label konflik (0 dan 1). Menghapus URL tersebut...")
    df_clean = df_raw[~conflict_mask].copy()
    print(f"[OK] Baris setelah pembersihan konflik: {len(df_clean):,}")
else:
    print("[OK] Tidak ditemukan konflik label pada dataset.")
    df_clean = df_raw.copy()

del df_raw
gc.collect()

# Pembuangan duplikat laten
initial_len = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['url_norm']).reset_index(drop=True)
dups_removed = initial_len - len(df_clean)
print(f"[OK] Berhasil menghapus {dups_removed:,} duplikat laten.")
print(f"[OK] Total baris dataset bersih dan unik: {len(df_clean):,}")

print("\n=== DISTRIBUSI LABEL SETELAH DEDUP ===")
print(df_clean['label'].value_counts())
print(df_clean['label'].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))


[INFO] Menjalankan fungsi normalisasi URL...
[OK] Baris setelah normalisasi: 1,559,362

[INFO] Mendeteksi URL dengan konflik label...
[PERINGATAN] Ditemukan 8 baris URL dengan label konflik (0 dan 1). Menghapus URL tersebut...
[OK] Baris setelah pembersihan konflik: 1,559,354
[OK] Berhasil menghapus 211,088 duplikat laten.
[OK] Total baris dataset bersih dan unik: 1,348,266

=== DISTRIBUSI LABEL SETELAH DEDUP ===
label
0    699057
1    649209
Name: count, dtype: int64
label
0    51.85%
1    48.15%
Name: proportion, dtype: object


## 5. Ekstraksi Domain Menggunakan `tldextract` & Penanganan Shared Hosting
1. Mengambil *registered domain* (`domain.suffix`).
2. Menerapkan penanganan khusus untuk domain *shared hosting* / *URL shortener* / *free web host* (misalnya: `google.com`, `github.io`, `bit.ly`, `wordpress.com`, `firebaseapp.com`, `web.app`, dll.) dengan menyertakan path tingkat pertama (`domain/path_1`) sebagai pembeda grup.


In [8]:
# Inisialisasi ekstraktor tldextract
extractor = tldextract.TLDExtract(cache_dir=None)

# Daftar domain shared hosting, shortener, dan platform publik
SHARED_DOMAINS = {
    'google.com', 'sites.google.com', 'docs.google.com', 'drive.google.com',
    'github.io', 'gitlab.io', 'bit.ly', 'tinyurl.com', 'wordpress.com',
    'blogspot.com', 'firebaseapp.com', 'web.app', 'weebly.com', 'wixsite.com',
    'azurewebsites.net', 'appspot.com', '000webhostapp.com', 'ngrok.io',
    'herokuapp.com', 'pages.dev', 'netlify.app', 'vercel.app', 'glitch.me',
    'replit.app', 's3.amazonaws.com'
}

def split_host_path(url_norm: str):
    parts = url_norm.split('/', 1)
    host = parts[0].split(':', 1)[0] # buang port jika ada
    first_path = parts[1].split('/')[0] if len(parts) > 1 and parts[1] else ""
    return host, first_path

print("[INFO] Memproses host dan path tingkat pertama...")
host_path_series = df_clean['url_norm'].apply(split_host_path)
df_clean['host'] = [hp[0] for hp in host_path_series]
df_clean['first_path'] = [hp[1] for hp in host_path_series]

# Cache tldextract pada unique hosts untuk efisiensi kecepatan
unique_hosts = df_clean['host'].unique()
print(f"[INFO] Mengekstrak {len(unique_hosts):,} host unik dengan tldextract...")

host_to_regdom = {}
for h in tqdm(unique_hosts, desc="tldextract"):
    ext = extractor(h)
    host_to_regdom[h] = ext.registered_domain.lower() if ext.registered_domain else h.lower()

df_clean['reg_domain'] = df_clean['host'].map(host_to_regdom)

def create_domain_group(row):
    reg_dom = row['reg_domain']
    host = row['host']
    path_1 = row['first_path']

    if (reg_dom in SHARED_DOMAINS or host in SHARED_DOMAINS) and path_1:
        return f"{reg_dom}/{path_1}"
    return reg_dom if reg_dom else host

print("[INFO] Membentuk domain_group dengan penanganan shared hosting...")
df_clean['domain_group'] = df_clean.apply(create_domain_group, axis=1)

# Bersihkan kolom bantu
df_clean = df_clean.drop(columns=['host', 'first_path', 'reg_domain'])

total_groups = df_clean['domain_group'].nunique()
print(f"\n[OK] Ekstraksi domain selesai!")
print(f"Total Sampel URL : {len(df_clean):,}")
print(f"Total Domain Grup: {total_groups:,}")
print("\n10 Domain Grup Terbanyak:")
print(df_clean['domain_group'].value_counts().head(10))


[INFO] Memproses host dan path tingkat pertama...
[INFO] Mengekstrak 327,895 host unik dengan tldextract...


tldextract:   0%|          | 0/327895 [00:00<?, ?it/s]

/tmp/ipykernel_1700/3189825483.py:32: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  host_to_regdom[h] = ext.registered_domain.lower() if ext.registered_domain else h.lower()


[INFO] Membentuk domain_group dengan penanganan shared hosting...

[OK] Ekstraksi domain selesai!
Total Sampel URL : 1,348,266
Total Domain Grup: 251,740

10 Domain Grup Terbanyak:
domain_group
blogspot.com/2011    17201
fc2.com              16471
blogspot.com/2010    15632
blogspot.com/2012    14198
blogspot.com/2009    12250
blogspot.com/2013    11607
blogspot.com/2014     9182
blogspot.com/2015     8059
blogspot.com/2008     7832
blogspot.com/2016     7157
Name: count, dtype: int64


## 6. Domain-Level Splitting (70% Train, 15% Val, 15% Test)
Menerapkan `GroupShuffleSplit` (`random_state=42`) dengan `groups=domain_group`:
- **Tahap 1**: Membagi seluruh dataset menjadi Train (70%) dan Temp (30%).
- **Tahap 2**: Membagi Temp (30%) secara simetris menjadi Val (15%) dan Test (15%) dengan `test_size=0.50`.


In [9]:
print("[INFO] Melakukan Domain-Level Splitting (70% Train, 15% Val, 15% Test)... ")

# Tahap 1: Train (70%) vs Temp (30%)
gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss_1.split(df_clean, groups=df_clean['domain_group']))

train_df = df_clean.iloc[train_idx].copy().reset_index(drop=True)
temp_df = df_clean.iloc[temp_idx].copy().reset_index(drop=True)

# Tahap 2: Temp (30%) -> Val (15%) & Test (15%)
gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss_2.split(temp_df, groups=temp_df['domain_group']))

val_df = temp_df.iloc[val_idx].copy().reset_index(drop=True)
test_df = temp_df.iloc[test_idx].copy().reset_index(drop=True)

del temp_df
gc.collect()

total_rows = len(df_clean)
print("\n=== HASIL PEMBAGIAN DATASET (DOMAIN SPLIT) ===")
print(f"Train Set : {len(train_df):,} baris ({len(train_df)/total_rows*100:.2f}%) | {train_df['domain_group'].nunique():,} domain grup")
print(f"Val Set   : {len(val_df):,} baris ({len(val_df)/total_rows*100:.2f}%) | {val_df['domain_group'].nunique():,} domain grup")
print(f"Test Set  : {len(test_df):,} baris ({len(test_df)/total_rows*100:.2f}%) | {test_df['domain_group'].nunique():,} domain grup")

print("\nDistribusi Label per Partisi:")
for name, dset in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    vc = dset['label'].value_counts()
    vcp = dset['label'].value_counts(normalize=True) * 100
    print(f" - {name:<5}: Benign(0) = {vc.get(0, 0):,} ({vcp.get(0, 0):.2f}%), Phishing(1) = {vc.get(1, 0):,} ({vcp.get(1, 0):.2f}%)")


[INFO] Melakukan Domain-Level Splitting (70% Train, 15% Val, 15% Test)... 

=== HASIL PEMBAGIAN DATASET (DOMAIN SPLIT) ===
Train Set : 927,898 baris (68.82%) | 176,218 domain grup
Val Set   : 198,052 baris (14.69%) | 37,761 domain grup
Test Set  : 222,316 baris (16.49%) | 37,761 domain grup

Distribusi Label per Partisi:
 - Train: Benign(0) = 478,671 (51.59%), Phishing(1) = 449,227 (48.41%)
 - Val  : Benign(0) = 100,051 (50.52%), Phishing(1) = 98,001 (49.48%)
 - Test : Benign(0) = 120,335 (54.13%), Phishing(1) = 101,981 (45.87%)


## 7. Zero-Leak Verification Test (Asersi Nol Kebocoran)
Memvalidasi secara ketat bahwa tidak ada satu pun domain atau URL normal yang bocor antar partisi data.


In [10]:
print("[INFO] Menjalankan Zero-Leak Verification Test...")

train_domains = set(train_df['domain_group'])
val_domains = set(val_df['domain_group'])
test_domains = set(test_df['domain_group'])

# 1. Asersi Train vs Test
overlap_train_test = train_domains.intersection(test_domains)
assert len(overlap_train_test) == 0, f"❌ LEAK DETECTED! Ditemukan {len(overlap_train_test)} domain overlap Train & Test!"
print("✅ [PASS] Asersi 1: Nol kebocoran domain antara Train dan Test (len overlap == 0).")

# 2. Asersi Train vs Val
overlap_train_val = train_domains.intersection(val_domains)
assert len(overlap_train_val) == 0, f"❌ LEAK DETECTED! Ditemukan {len(overlap_train_val)} domain overlap Train & Val!"
print("✅ [PASS] Asersi 2: Nol kebocoran domain antara Train dan Val (len overlap == 0).")

# 3. Asersi Val vs Test
overlap_val_test = val_domains.intersection(test_domains)
assert len(overlap_val_test) == 0, f"❌ LEAK DETECTED! Ditemukan {len(overlap_val_test)} domain overlap Val & Test!"
print("✅ [PASS] Asersi 3: Nol kebocoran domain antara Val dan Test (len overlap == 0).")

# 4. Asersi Konsistensi Jumlah Total Baris
assert len(train_df) + len(val_df) + len(test_df) == len(df_clean), "❌ Jumlah total baris tidak konsisten!"
print(f"✅ [PASS] Asersi 4: Konsistensi total baris ({len(df_clean):,} = {len(train_df):,} + {len(val_df):,} + {len(test_df):,}).")

# 5. Asersi String URL Normal Bebas Kebocoran
train_urls = set(train_df['url_norm'])
val_urls = set(val_df['url_norm'])
test_urls = set(test_df['url_norm'])
assert len(train_urls.intersection(test_urls)) == 0, "❌ URL leak antara Train & Test!"
assert len(train_urls.intersection(val_urls)) == 0, "❌ URL leak antara Train & Val!"
assert len(val_urls.intersection(test_urls)) == 0, "❌ URL leak antara Val & Test!"
print("✅ [PASS] Asersi 5: Nol kebocoran URL string antar seluruh partisi.")

print("\n🎉 SELURUH VERIFIKASI ZERO-LEAK BERHASIL 100%! DATASET BERSIH & ROBUST.")


[INFO] Menjalankan Zero-Leak Verification Test...
✅ [PASS] Asersi 1: Nol kebocoran domain antara Train dan Test (len overlap == 0).
✅ [PASS] Asersi 2: Nol kebocoran domain antara Train dan Val (len overlap == 0).
✅ [PASS] Asersi 3: Nol kebocoran domain antara Val dan Test (len overlap == 0).
✅ [PASS] Asersi 4: Konsistensi total baris (1,348,266 = 927,898 + 198,052 + 222,316).
✅ [PASS] Asersi 5: Nol kebocoran URL string antar seluruh partisi.

🎉 SELURUH VERIFIKASI ZERO-LEAK BERHASIL 100%! DATASET BERSIH & ROBUST.


## 8. Simpan Hasil Split ke Google Drive
Menyimpan `train.csv`, `val.csv`, dan `test.csv` ke direktori Google Drive: `/content/drive/MyDrive/Skripsi/new data/`.


In [11]:
print(f"[INFO] Menyimpan partisi dataset ke Google Drive: {DATA_DIR}...")

export_cols = ['url', 'url_norm', 'domain_group', 'label']

# Simpan train.csv
t0 = time.perf_counter()
train_df[export_cols].to_csv(TRAIN_CSV_PATH, index=False)
train_mb = os.path.getsize(TRAIN_CSV_PATH) / (1024 * 1024)
print(f"✅ train.csv tersimpan : {TRAIN_CSV_PATH} ({train_mb:.2f} MB | {time.perf_counter() - t0:.2f} dtk)")

# Simpan val.csv
t0 = time.perf_counter()
val_df[export_cols].to_csv(VAL_CSV_PATH, index=False)
val_mb = os.path.getsize(VAL_CSV_PATH) / (1024 * 1024)
print(f"✅ val.csv tersimpan   : {VAL_CSV_PATH} ({val_mb:.2f} MB | {time.perf_counter() - t0:.2f} dtk)")

# Simpan test.csv
t0 = time.perf_counter()
test_df[export_cols].to_csv(TEST_CSV_PATH, index=False)
test_mb = os.path.getsize(TEST_CSV_PATH) / (1024 * 1024)
print(f"✅ test.csv tersimpan  : {TEST_CSV_PATH} ({test_mb:.2f} MB | {time.perf_counter() - t0:.2f} dtk)")

print("\n=== RINGKASAN FILE DI GOOGLE DRIVE ===")
for f_name in ["train.csv", "val.csv", "test.csv"]:
    f_path = os.path.join(DATA_DIR, f_name)
    size_mb = os.path.getsize(f_path) / (1024 * 1024)
    print(f" 📁 {f_name:<12} : {size_mb:6.2f} MB")

print("\nPratinjau 5 baris pertama train.csv:")
display(pd.read_csv(TRAIN_CSV_PATH, nrows=5))

print("\n🚀 [SELESAI] Pipeline Data Preparation selesai dengan sukses!")


[INFO] Menyimpan partisi dataset ke Google Drive: /content/drive/MyDrive/Skripsi/new data...
✅ train.csv tersimpan : /content/drive/MyDrive/Skripsi/new data/train.csv (138.84 MB | 5.59 dtk)
✅ val.csv tersimpan   : /content/drive/MyDrive/Skripsi/new data/val.csv (30.46 MB | 1.35 dtk)
✅ test.csv tersimpan  : /content/drive/MyDrive/Skripsi/new data/test.csv (33.44 MB | 1.41 dtk)

=== RINGKASAN FILE DI GOOGLE DRIVE ===
 📁 train.csv    : 138.84 MB
 📁 val.csv      :  30.46 MB
 📁 test.csv     :  33.44 MB

Pratinjau 5 baris pertama train.csv:


,url,url_norm,domain_group,label
0,http://bs-369.com/Views/Account/Login.aspx?Ret...,bs-369.com/views/account/login.aspx?returnurl=,bs-369.com,1
1,http://amelihair.ru/images/6550054213/74535376...,amelihair.ru/images/6550054213/745353769679,amelihair.ru,1
2,https://www.behance.net/gallery/23251525/Self-...,behance.net/gallery/23251525/self-branding,behance.net,0
3,http://ssl32.sslbr.co.vu/gol/,ssl32.sslbr.co.vu/gol,co.vu,1
4,http://lakoner.com/loa/verify/,lakoner.com/loa/verify,lakoner.com,1



🚀 [SELESAI] Pipeline Data Preparation selesai dengan sukses!
